# 02 - Calibration of the Player Generation Model

**Objective**: This notebook aims to extract some statistical laws underlying the carreers of tennis players from ATP results (1991-2024). We will extract parameters from our model to configure the players in our simulation later. 

**Method 1**: Each player will be attributed a score _S_ determined by an _intrinsic potential P_, calibrated on the distribution of maximum strengths of the players (based on the dataset available), and _aging A_, that is a function describing the evolution of strength with age.

## 0. Creation of Players Database Info

We create a dataset containing the main information about players' careers. We extract the maximum strength achieved by each player (from the `zermelo_strengths_1991-2024.csv` file) and the age at which this maximum strength was reached. We also store the age at first and last match played (We use the year of birth from the `atp_players.csv` file to compute ages). 

Since age is a crucial factor in our model, any player with missing birthdate information is excluded from the dataset. (The impact is minimal, as these kind of players have not played many matches in the dataset and have small Zermelo strengths).

For the players active in the boundaries of the dataset (1991 and 2024), we retain them in the dataset because they provide valuable information about the distribution of strengths. However, they will be excluded from the calibration of aging curves and carreer duration later, since we do not have their full career data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

# Note: scipy.stats is used throughout the code.

In [ ]:
zermelo_strengths_data = pd.read_csv("../data/processed/zermelo_strengths_1991-2024.csv")
players_info_data = pd.read_csv("../data/tennis_atp/atp_players.csv", low_memory=False)
#display(players_info_data.head())

In [ ]:
# sort the data so we can get the year corresponding to the maximum strength on the first line
zermelo_strengths_data_sorted = zermelo_strengths_data.sort_values("zermelo_strength", ascending=False)

# group the data by IDs, and then get: start and end years, max strength and corresponding year
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html
players_stats_data = zermelo_strengths_data_sorted.groupby("player_id").agg(start_year=("year", "min"), 
                                                       end_year=("year", "max"), 
                                                       top_year=("year", "first"),
                                                       top_strength=("zermelo_strength", "max"),
                                                       active_years=("year", "nunique")).reset_index()                     

display(players_stats_data)

In [ ]:
# formatting year of birth to get only the year (with .dt.year)
# https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
# https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior
players_info_data["birth_year"] = pd.to_datetime(players_info_data["dob"].astype(str), format="%Y%m%d.0", errors="coerce").dt.year

In [ ]:
players_full_data = pd.merge(players_stats_data, players_info_data[["player_id","birth_year", "name_first", "name_last"]], on="player_id", how="left")
players_full_data = players_full_data.dropna(subset = ["birth_year"]) # delete all the players without a valid birth year 
players_full_data["birth_year"] = players_full_data["birth_year"].astype(int) # to remove the .0 after the year

# add start_age / top_age / end_age
players_full_data["start_age"]= players_full_data["start_year"]-players_full_data["birth_year"]
players_full_data["end_age"]= players_full_data["end_year"]-players_full_data["birth_year"]
players_full_data["top_age"]= players_full_data["top_year"]-players_full_data["birth_year"]

# removal of players who have a negative age or are younger than 14 years old
# It seems that different players have the same ID, so they are deleted (e.g. son confused with his father)
# example: Martin Damm (father born in 1972, son born in 2003 --> year displayed: 2003)
players_full_data = players_full_data[players_full_data["start_age"]>=14]

display(players_full_data.head())

In [ ]:
#check whether a player was active both in 1991 and in 2024
all_years_player = players_full_data[(players_full_data["start_year"]==1991) & (players_full_data["end_year"]==2024)]
print("Number of players active both in 1991 and in 2024:", len(all_years_player))

# Detecting complete careers: new column added in the data to find the players whose start and end are available 
# (i.e. who started after 1991 and ended before 2024)

players_full_data["career_type"] = np.where((players_full_data["start_year"]>1991) & 
                                                (players_full_data["end_year"]<2024), 
                                                "full", "Other")

players_full_data["career_type"] = np.where(players_full_data["start_year"]==1991, "left_boundary",
                                               players_full_data["career_type"])

players_full_data["career_type"] = np.where(players_full_data["end_year"]==2024, "right_boundary",
                                               players_full_data["career_type"])


full_career_players_nbr = (players_full_data["career_type"]=="full").sum() 
valid_careers_nbr = len(players_full_data)

print("Number of complete careers:", full_career_players_nbr, "out of", valid_careers_nbr, 
      f"({100*full_career_players_nbr/valid_careers_nbr:.2f}%)")

In [ ]:
# rearranging the order of the columns
players_full_data = players_full_data[["player_id", "name_first", "name_last", 
                                       "birth_year", "start_year", "end_year", "top_year", 
                                       "start_age", "end_age", "top_age", 
                                       "top_strength", "career_type", "active_years"]]

In [ ]:
# saving the file in a .csv
output_path = "../data/processed/players_stats.csv"
players_full_data.to_csv(output_path, index=False)

All the information is stored in a new dataframe `players_full_data`. It contains the following columns: $\\$
$\textbf{player\_id, name\_first, name\_last, birth\_year, start\_year, end\_year, top\_year, start\_age}$
$\textbf{end\_age, top\_age, top\_strength, complete\_career}$.

## 1. Testing the Stationarity of the Player Generation Process

Before calibrating our model, we must verify that the process generating players is stationary over time. This verification stands on the hypothesis that the distribution of players' intrinsic potentials (maximum strengths) remains consistent across different time periods, and that the number of new players entering the professional circuit each year is relatively stable. 

### 1.1. Stationarity of Maximum Strengths Distribution (Quality)

We first check that the distribution of players' strengths is stationary over time. We plot the distribution of players' maximum strengths for different time periods and compare them. To validate this hypothesis, we examine the evolution of `top_strength` based on the player's `start_year` by focusing on the average level stability (median) and the elite level stability (top 10% players). The overall shape of distribution over decades is also analysed to ensure no significant shifts occur.

If this hypothesis holds, we can pool all players together to estimate the distribution of intrinsic potentials.

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})


# plot of the median trend line
sns.lineplot(data=players_full_data,
             x="start_year", y="top_strength", 
             estimator="median", errorbar=("pi",50), # colored band containing 50% of the points around the median
             color="#e74c3c",linewidth=3,
             label="Median Strength",
             zorder=3)

# plot of the top 10% trend line
sns.lineplot(data=players_full_data, 
             x="start_year", y="top_strength", 
             estimator=lambda x: np.percentile(x, 90), # 90th percentile = top 10% (value )
             errorbar=("ci", 95), # confidence interval of 95%
             label="90th Percentile Strength",
             color="#8e44ad", linewidth=3, linestyle="--", zorder=4)

# plot of the maximum trend line
sns.lineplot(data=players_full_data, x="start_year", y="top_strength", 
             estimator=np.max, errorbar=None, 
             label="Max Strength",
             color="#f1c40f", linewidth=3, linestyle="-.", zorder=5)


# plot of the individual points
sns.stripplot(data=players_full_data, 
              x="start_year", y="top_strength", 
              size=2.5, 
              hue="career_type",
              hue_order=["full", "left_boundary", "right_boundary"],
              palette=["#687475", "#60a215e8", "#1484cf"], 
              jitter=0.3, #to avoid overplotting
              native_scale=True, 
              zorder=2)

plt.title("Distribution of Player Maximum Zermelo Strengths by Career Start Year", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12)

plt.yscale("log")
plt.ylabel(r"Maximum Zermelo Strength ($\pi_{max}$)", fontsize=18)

# legend (keep only the relevant legend items)
handles, _ = plt.gca().get_legend_handles_labels()

handles_curves = [handles[0], handles[1], handles[2]]
labels_curves = ["Median (IQR band)", "Top 10% (95% CI)", "Top 1"]
legend_curves = plt.legend(handles=handles_curves, 
                           labels=labels_curves, 
                           markerscale=2.5, fontsize=13,
                           loc="upper right",
                           frameon=False)

plt.gca().add_artist(legend_curves) # to keep both legends


handles_points = [handles[4], handles[3], handles[5]]
labels_points = ["Left bounded (start in 1991)", "Complete Career", "Right bounded (end in 2024)"]
plt.legend(handles=handles_points, 
                           labels=labels_points, 
                           markerscale=3, fontsize=13,
                           bbox_to_anchor=(0.8, -0.15),
                           frameon=True, ncol=3)


plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

<font color="blue"> **_The figure strongly supports our stationary hypothesis!_** </font> The median player strength (in red) and the elite threshold (in purple) remain horizontal and stable from 1992 to 2018, proving that the generation of talent has not changed structurally over time. 

The high variance in 1991 comes from players already in their prime (they started before 1991, but the first year of our dataset is 1991). We observe a slight increase in 2019, immediately followed by a decrease in 2020 (likely due to consequences of the COVID-19 pandemic). After this year, it continues decreasing due to right-censoring: players starting their carreers recently have not yet reached their real maximum strength. 

Regarding the elites, the top 10% threshold mirrors the stabilitiy of the median, except with some small fluctuations. It confirms that the top players can be generated consistently year over year. 

The Top 1 curve (in yellow) is more volatile, with peaks rather than a specific trend. This is expected, as the very best players can vary significantly from year to year due to the emergence of exceptional talents or the dominance of a few players.

---

To ensure robutstness, we must restrict the calibration of our model parameters to a stable period **1992-20YY**, where the stationary hypothesis is most valid. To determine the cut-off year, we estimate the time required for players to reach their potential, by taking players that started between 1992 and 2002.  It will be done by calcutating the years to reach their maximum strength. 

In [ ]:
safe_year_start = 1992
safe_year_end = 2002

In [ ]:
# finding the players that begin between 1992 and 2002
years_to_top_players = players_full_data[(players_full_data["start_year"]>=safe_year_start) & (players_full_data["start_year"]<=safe_year_end)].copy()

# calculating the years to reach top
years_to_top_players["years_to_top"] = years_to_top_players["top_year"]-years_to_top_players["start_year"]

years_to_top_players["years_to_top"].describe(percentiles=[0.25,0.5,0.9, 0.95, 0.99])

The median tie to reach peak strength is only 1 year, and the first quartile is 0 year. This indicates that the majority of players do not experience a long development: they enter the tour, reach their maximum almost immediately and decline. As a result, their top year is simply the year they retired, not their true potential. 

<font color="blue"> **To fix this problem, we filter the data by keeping only players who were active for at least some years.** </font>

In [ ]:
min_active_years = 5

In [ ]:
filtered_years_to_top_players = years_to_top_players[years_to_top_players["active_years"]>= min_active_years].copy()
print(f"Number of players with at least {min_active_years} active years ({safe_year_start}-{safe_year_end}):", len(filtered_years_to_top_players))

In [ ]:
# calculating the years to reach top
filtered_years_to_top_players["years_to_top"] = filtered_years_to_top_players["top_year"]-filtered_years_to_top_players["start_year"]

filtered_years_to_top_players["years_to_top"].describe(percentiles=[0.25,0.5,0.9, 0.95, 0.99])

The maximum value of 22 years seems to be unrealistic. The 90th percentile lies at 10 years. This means that only 10% of players take more than 10 years to reach their peak, which is a reasonable threshold to consider for our cut-off year.

<font color="purple"> Calibration is made on the starting year period **1992-2014**! </font> 
$\newline$ _to ensure that all players (the vast majority) have reached their potential._

In [ ]:
calibration_start = 1992
calibration_end = 2014

Now, we want to see if the distribution of maximum strengths is consistent across generations. We plot the distribution of `top_strength` for players starting in different decades (1990s, 2000s, 2010s).

In [ ]:
# considering only the players in the calibration period
calibration_players = players_full_data[(players_full_data["start_year"]>=calibration_start) & 
                                        (players_full_data["start_year"]<=calibration_end) &
                                        (players_full_data["active_years"] >= min_active_years)].copy()

print("Total number of players in the calibration period:", len(calibration_players))

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})


# plot of the median trend line
sns.lineplot(data=calibration_players,
             x="start_year", y="top_strength", 
             estimator="median", errorbar=("pi",50), # colored band containing 50% of the points around the median
             color="#e74c3c",linewidth=3,
             label="Median Strength",
             zorder=3)

# plot of the top 10% trend line
sns.lineplot(data=calibration_players, 
             x="start_year", y="top_strength", 
             estimator=lambda x: np.percentile(x, 90), # 90th percentile = top 10% (value )
             errorbar=("ci", 95), # confidence interval of 95%
             label="90th Percentile Strength",
             color="#8e44ad", linewidth=3, linestyle="--", zorder=4)

# plot of the maximum trend line
sns.lineplot(data=calibration_players, x="start_year", y="top_strength", 
             estimator=np.max, errorbar=None, 
             label="Max Strength",
             color="#f1c40f", linewidth=3, linestyle="-.", zorder=5)


# plot of the individual points
sns.stripplot(data=calibration_players, 
              x="start_year", y="top_strength", 
              size=2.5, 
              hue="career_type",
              hue_order=["full", "left_boundary", "right_boundary"],
              palette=["#687475", "#60a215e8", "#1484cf"], 
              jitter=0.3, #to avoid overplotting
              native_scale=True, 
              zorder=2)

plt.title("Distribution of Player Maximum Zermelo Strengths by Career Start Year \n(Min. 5 Active Years)", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12)

plt.yscale("log")
plt.ylabel(r"Maximum Zermelo Strength ($\pi_{max}$)", fontsize=18)

# legend (keep only the relevant legend items)
handles, _ = plt.gca().get_legend_handles_labels()

handles_curves = [handles[0], handles[1], handles[2]]
labels_curves = ["Median (IQR band)", "Top 10% (95% CI)", "Top 1"]
legend_curves = plt.legend(handles=handles_curves, 
                           labels=labels_curves, 
                           markerscale=2.5, fontsize=13,
                           loc="upper right",
                           frameon=False)

plt.gca().add_artist(legend_curves) # to keep both legends


handles_points = [handles[4], handles[3], handles[5]]
labels_points = ["Left bounded (start in 1991)", "Complete Career", "Right bounded (end in 2024)"]
plt.legend(handles=handles_points, 
                           labels=labels_points, 
                           markerscale=3, fontsize=13,
                           bbox_to_anchor=(0.8, -0.15),
                           frameon=True, ncol=3)


plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# getting the decades of the players (1990s, 2000s, 2010s)
calibration_players["generation"] = calibration_players["start_year"] - (calibration_players["start_year"] % 10)
#display(calibration_players["generation"].value_counts())

# creation of the labels for the generations
generations_min = calibration_players.groupby("generation")["start_year"].min()
generations_max = calibration_players.groupby("generation")["start_year"].max()

labels = []

for generation in generations_min.index:
    start = generations_min[generation]
    end = generations_max[generation]
    label = f"{generation}s ({start}-{end})" if start != end else f"{generation}s ({start})"
    labels.append(label)

calibration_players["generation_label"] = calibration_players["generation"].replace(generations_min.index, labels)

calibration_players = calibration_players.sort_values("generation")
hue_order = calibration_players["generation_label"].unique()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plot of each distribution by generation
sns.kdeplot(data=calibration_players, 
            x="top_strength", 
            hue="generation_label", 
            hue_order=hue_order,
            log_scale=True, fill=False,
            common_norm=False,
            linewidth=4, alpha=1,
            palette="viridis",
            zorder=2)

# plot of the global distribution (all players) if show_global is True
show_global = False

if show_global:
    sns.kdeplot(data=calibration_players, 
                x="top_strength",
                log_scale=True, fill=False,
                common_norm=False,
                linewidth=2.5, alpha=0.9,
                color="red",
                linestyle="--",
                zorder=3)

plt.title("Maximum Zermelo Strength Distribution by Generation", fontsize=25, weight="bold", pad=35)

plt.xlabel(r"Maximum Zermelo Strength ($\pi_{max}$)", fontsize=18)
plt.xticks(fontsize=12)

plt.ylabel("Density", fontsize=18)

# legend
handles = plt.gca().get_lines()
all_labels = list(hue_order)

if show_global:
    all_labels.append("Global Distribution")


legend = plt.legend(handles=handles, title="Career Start Decade", 
                    labels=all_labels, 
                    fontsize=13, title_fontsize=14, 
                    loc="upper center", frameon=False)
plt.setp(legend.get_title(), fontweight='bold')


plt.grid(visible=True, which="major", axis="x", color="gray", linewidth=0.5, alpha=0.5)
plt.grid(visible=True, which="minor", axis="x", color="gray", linestyle=':', linewidth=0.5, alpha=0.3)

sns.despine()
plt.tight_layout()
plt.show()

The KDE plot reveals similarity across all three decades. The distributions share a similar shape, with peaks around the same strengths, followed by a heavy right tail. While the blue curve (2010s) appears a bit shifted and more volatile, this is likely due to more money and opportunities in recent years, leading to more good players. However, the overall shape remain consistent, which supports the stationarity hypothesis.

In [ ]:
stats = calibration_players.groupby("generation_label")["top_strength"].describe()

stats[["count", "mean", "50%", "std", "max"]]

The `mean` and `50%` (median) value is quite stable across decades, with a slight decrease in the 2010s. 

By looking at the `std` column, it suggests that the system is chaotic. It drops significantly from the 2000s to the 2010s (by a factor of 3)! It would mean that the distributions changed a lot, but here it simply means that linear statistics are not sufficient to capture the distribution of strengths. There is a correlation between the maximum strength and the standard deviation!

The standard deviation is dominated by the right tail of the distribution, and is influenced a lot by the presence of a few very strong players. We should move to the logarithmic scale:

In [ ]:
calibration_players["log10_strength"] = np.log10(calibration_players["top_strength"])

log_stats = calibration_players.groupby("generation_label")["log10_strength"].describe()

log_stats[["count", "mean", "50%", "std", "max"]]

When analysed in the log scale, the standard deviation is much more stable across decades, confirming that the apparent increase in variability was due to the presence of a few outliers in the right tail of the distribution. 

The log transformation helps to stabilize the variance and helps for finding theunderlying distribution of player strengths and confirming that the generation process has remained consistent over time.

<font color="purple"> _Conclusion_: **We can therefore validate the stationary hypothesis of maximum strength distribution**, and pool all players together to estimate the distribution of intrinsic potentials. We will use the logarithm of `top_strength` for the calibration of our model parameters. </font>

In [ ]:
# saving the calibration players data in a .csv

output_calibration_path = f"../data/processed/calibration_players_{calibration_start}-{calibration_end}.csv"
calibration_players.to_csv(output_calibration_path, index=False)

### 1.2. Stationarity of the Number of New Players (Quantity)

To ensure having a realistic simulation, we need to model the arrival of new players. Instead of a fixed number, we could the incoming flux using a Gaussian distribution $\mathcal{N}(\mu, \sigma)$, where $\mu$ is the average number of new players per year and $\sigma$ is the standard deviation of the number of new players per year. The other possibility is a linear regression to see whether there is a significant trend.

In [ ]:
# counting the number of players starting each year in the calibration period
numbers_per_year = calibration_players.groupby("start_year")["start_year"].count()

In [ ]:
# calculating the mean and standard deviation of the number of new players per year
arrival_mean = numbers_per_year.mean()
arrival_std = numbers_per_year.std()

print(f"Average number of new players per year (\u03bc): {arrival_mean:.1f}")
print(f"Standard deviation of new players per year (\u03c3): {arrival_std:.1f}")

In [ ]:
# histogram of the number of new players per year

plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

sns.barplot(x=numbers_per_year.index.values, y=numbers_per_year.values, color="#3498db", 
            alpha=0.6, zorder=2, edgecolor="black")

# horizontal line for the mean value
plt.axhline(y=arrival_mean, 
            color="#e74c3c", label=rf"Mean value: $\mu_A$={arrival_mean:.1f}", 
            zorder=3, linewidth=3, linestyle="-")

# zone containing the mean value +/- 2 standard deviations (95% of the data if normal distribution)
plt.axhspan(ymin=arrival_mean - 2*arrival_std, 
            ymax=arrival_mean + 2*arrival_std, 
            color="#e74c3c", alpha=0.1, label=rf"95% CI ($\mu_A \pm 2\sigma_A$, $\sigma_A = {arrival_std:.1f}$)", zorder=0)

plt.title(f"Flux of New Players Entering the Tour per Year ({calibration_start}-{calibration_end})", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12, ticks=range(0, len(numbers_per_year),3))

plt.ylabel("Number of New Players ($N_{new}$)", fontsize=18)


plt.grid(visible=True, axis="y", linewidth=0.5, alpha=0.7, zorder=0)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

#### H1: Gaussian Distribution Guess

We see that $\sigma$ >> $\sqrt{\mu}$ $(\approx 26.1$), which means that a Poisson distribution would not be appropriate there. To see if the Gaussian distribution is a good fit, we can do a Shapiro-Wilk test (more robust than the Kolmogorov-Smirnov test for small samples) for normality. $\textbf{The null hypothesis is that the data is normally distributed.}$ If the p-value is greater than a significance level (e.g., 0.05), we fail to reject the null hypothesis, suggesting that the data is consistent with a normal distribution. But this test is not sufficient, so we will also look at the histogram and the Q-Q plot of the data to visually assess the normality.

We observed on the previous graph that the shaded area representing the 95% confidence interval (mean ± 2 standard deviations) covers most of the data points, which is consistent with the properties of a normal distribution.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html
from scipy.stats import shapiro

shapiro_stat_normal, shapiro_p_value_normal = shapiro(numbers_per_year)

print(f"Shapiro-Wilk Test Statistic: {shapiro_stat_normal:.4f}")
print(f"Shapiro-Wilk Test p-value: {shapiro_p_value_normal:.4f}")

if shapiro_p_value_normal > 0.05:
    print("\nFail to reject the null hypothesis: the data is consistent with a normal distribution.")
else:
    print("\n Reject the null hypothesis: the data is not consistent with a normal distribution.")

To confirm visually the result of the test (as the number of data points is quite small), we use a Q-Q Plot:
 
- X axis: Theoretical quantiles from  a standard normal distribution $\mathcal{N}(0,1)$, corresponding to a division of the data into $n$ slices of equal probability (where $n$ is the number of data points). For each slice $i$, we calculate $P_i$=$(i-0.5)/n$, which gives us the cumulative probability up to that slice. We then convert it into a Z-score using the probit function (the inverse of the cumulative distribution function): $Z_i$ = $\Phi^{-1}(P_i)$. It representes the theoretical distance from the mean measured in units of standard deviation.


- Y axis: The actual data points, sorted in ascending order. Each point corresponds to the number of new players in a given year, ordered from the smallest to the largest.

- Red line: The line represents the expected relationship if the data were perfectly normally distributed, i.e. Y= $\mu + \sigma X$. If the points closely follow this line, it suggests that the data is consistent with a normal distribution.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html
from scipy.stats import probplot

# osm and osr: tuple of theoretical quantiles and ordered values of the data, used for plotting the Q-Q plot
# slope and intercept and r: standard deviation, mean and correlation coefficient of the data (used for the red line in the Q-Q plot)
(osm_normal, osr_normal), (slope_normal, intercept_normal, r_normal) = probplot(numbers_per_year, dist="norm")

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plot of points in the Q-Q plot
plt.scatter(osm_normal, osr_normal, color="#3498db", edgecolor="black", s=200, zorder=3)

# plot of the red line representing the expected relationship if the data were perfectly normally distributed
x_line = np.array([min(osm_normal), max(osm_normal)])
y_line = intercept_normal + slope_normal * x_line
plt.plot(x_line, y_line, color="#e74c3c", linewidth=3, zorder=2,
         label=r"Normal Reference Line ($Y = \mu_A + \sigma_A X$)")



plt.title("Normal Q-Q Plot of the Number of New Players per Year", fontsize=25, weight="bold", pad=35)
plt.xlabel("Theoretical Quantiles X (Standard Z-scores)", fontsize=18)
plt.ylabel("Data Quantiles Y (Number of New Players)", fontsize=18)


plt.grid(visible=True, axis="x", color="gray", linewidth=0.5, alpha=0.3)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper left")
plt.tight_layout()
plt.show()

This graph shows a strong linear trend. Most of the data points are closely aligned with the red reference line, particularly in the $[-1.5,1.5]$ range. This indicates that the core of the distribution matches the Normal model very well. We observe slight deviations at the extremities, especially for the maximum value. It suggest that these extreme values occur more frequently in reality than a perfect normal distribution would predict.

Finally, we can calculate the coefficient of determination $R^2$, which measures the goodness of fit of our data to the normal data. Here, it is the correlation between the theoretical normal quantiles and the observed quantiles. A value close to 1 indicates that the normal distribution is a highly accurate representation of the data.  

In [ ]:
print(f"R^2 = {r_normal**2:.3f}")

#### H2: Linear Regression Guess

The hypothesis of a normal distribution seems to be okay, but the graph at the beginning of $\textit{Section 1.2}$ suggests that there might be a slight increasing trend in the number of new players over time. So let's try to do a linear regression and get the slope and intercept for the number of new players per year, to see whether it is significant or not.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html
from scipy.stats import linregress

# extract the years and the incoming number of players (as well as coefficient determination R², p-value, and )
years = numbers_per_year.index.values
years_shift = years - calibration_start # so that the regression is done from 0 to len(years) and not from calibration_start to calibration end

incoming_players = numbers_per_year.values

incoming_players_regression = linregress(years_shift,incoming_players)

print(f"Slope: {incoming_players_regression.slope:.2f} ± {incoming_players_regression.stderr:.2f}")
print(f"Coefficient of determination: R^2 = {incoming_players_regression.rvalue**2:.2f}")
print(f"p-value: {incoming_players_regression.pvalue:.2e}")

if incoming_players_regression.pvalue < 0.05:
    print("\n There is a significant trend in the number of new players per year!")
else:
    print("\n There is no significant trend in the number of new players per year!")

In [ ]:
# calculating the fitted values for the number of new players per year (based on the linear regression)
fit_new_players = incoming_players_regression.slope*years_shift + incoming_players_regression.intercept

In [ ]:
# histogram of the number of new players per year

plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

sns.barplot(x=numbers_per_year.index.values, y=numbers_per_year.values, color="#3498db", 
            alpha=0.6, edgecolor="black", zorder=1)

plt.plot(range(len(years)), fit_new_players, 
         label=f"Linear Fit: $N_{{new}}(t_0) = {incoming_players_regression.slope:.1f}(t_0 - {calibration_start}) + {incoming_players_regression.intercept:.0f}$", 
         color="#e74c3c", zorder=3, linewidth=5, linestyle="-")


plt.title(f"Flux of New Players Entering the Tour per Year ({calibration_start}-{calibration_end})", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12, ticks=range(0, len(numbers_per_year),3))

plt.ylabel("Number of New Players ($N_{new}$)", fontsize=18)


plt.grid(visible=True, axis="y", linewidth=0.5, alpha=0.7, zorder=0)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

The linear regression confirms a significant upward trend (p-value $<0.05$). The hypothesis of stationarity for the quantity of incoming players is then rejected. However, a linear equation predicts exactly the number of players and lacks the volatility of the real world. To built a realistic simulation, we give some "noise" around this trend. 

We can use a Gaussian distribution with mean 0 and standard deviation equal to the standard deviation of the differences between the real number of incoming players and the predicted number by the regression (residuals). 

Let's see if it is a good choice, by stating that the null hypothesis is that the differences between the real number of incoming players and the predicted number by the regression are normally distributed with mean 0 and standard deviation equal to the standard deviation of the differences. We can use the Shapiro-Wilk test to test this hypothesis, as well as a Q-Q plot.

In [ ]:
# calculate the difference between the real number of incoming players and the predicted number by the regression (residuals)
difference_new_players = fit_new_players - incoming_players

# calculate the mean and standard deviation of the difference

residuals_mean = difference_new_players.mean()
residuals_std = difference_new_players.std()

print(f"Mean of the residuals (difference between real and predicted number of new players): {residuals_mean:.2e}")
print(f"Standard deviation of the residuals: {residuals_std:.2f}")

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html
from scipy.stats import shapiro

shapiro_stat_regression, shapiro_p_value_regression = shapiro(difference_new_players)

print(f"Shapiro-Wilk Test Statistic: {shapiro_stat_regression:.4f}")
print(f"Shapiro-Wilk Test p-value: {shapiro_p_value_regression:.4f}")

if shapiro_p_value_regression > 0.05:
    print("\nFail to reject the null hypothesis: the data is consistent with a normal distribution.")
else:
    print("\n Reject the null hypothesis: the data is not consistent with a normal distribution.")


In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html
from scipy.stats import probplot

# osm and osr: tuple of theoretical quantiles and ordered values of the data, used for plotting the Q-Q plot
# slope and intercept and r: standard deviation, mean and correlation coefficient of the data (used for the red line in the Q-Q plot)
(osm_regression, osr_regression), (slope_regression, 
                                   intercept_regression, r_regression) = probplot(difference_new_players, dist="norm")

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plot of points in the Q-Q plot
plt.scatter(osm_regression, osr_regression, color="#3498db", edgecolor="black", s=200, zorder=3)

# plot of the red line representing the expected relationship if the data were perfectly normally distributed
x_line = np.array([min(osm_regression), max(osm_regression)])
y_line = intercept_regression + slope_regression * x_line
plt.plot(x_line, y_line, color="#e74c3c", linewidth=3, zorder=2,
         label=r"Normal Reference Line ($Y = 0 + \sigma_{{res}} X$)")



plt.title("Normal Q-Q Plot of the Residuals", fontsize=25, weight="bold", pad=35)
plt.xlabel("Theoretical Quantiles X (Standard Z-scores)", fontsize=18)
plt.ylabel("Data Quantiles Y (Residuals)", fontsize=18)


plt.grid(visible=True, axis="x", color="gray", linewidth=0.5, alpha=0.3)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
print(f"R^2 = {r_regression**2:.3f}")

Visually, it shows great alignment with the red theoretical line, and the SW test yields a p-value well above 0.05. It proves that fluctuations could be modeled by a Gaussian distribution!

#### Conclusion for the Generation of New Players each year

Based on this analysis, we now have 2 possible mathematical models to generate the influx of new players $N_{new}(t)$ for year $t$ in the simulation. Note that we can't generate a non-integer number of players, so the final result will be rounded to the nearest integer.


**1. Stationary Model (Gaussian distribution)**

This is the simplest model, assuming the generation is constant over time. The distribution depends only on the average $\mu_A$, with standard deviation $\sigma_A$ (calculated from the available data):

$$N_{new}(t) = \text{Round}(X_t), \quad X_t \sim \mathcal{N}(\mu_A, \sigma_A^2)$$

_(Remark: it ignores the expansion of the number of new players, and understimates the number of players in a long future.)_

**2. Dynamic Model (Linear Regression + "Gaussian Noise")**

This model combines the upward expansion of the tour with normally distributed residuals (noise for more reality). This means that:

$$N_{new}(t) = \text{Round} \left( \alpha_A \cdot (t - t_{ref}) + \beta_A + \epsilon_t \right), \quad \epsilon_t \sim \mathcal{N}(0, \sigma_{res}^2)$$

where $t_{ref}$ the reference year of the calibration (e.g. 1992), $\alpha_A$ and $\beta_A$ respectively the slope (new number of players each year) and intercept (number of new players at $t_{ref}$) of the regression. Here, $\epsilon_t$ is the "noise" added based on the standard deviation $\sigma_{res}$ of the residuals.

#### Saving the parameters

For completeness, we save the parameters of both the stationary model and the dynamic one (with increase of number of players with years) in a $\textit{.json}$ file.

In [ ]:
# saving the arrival parameters (mean and std) in .json file

arrival_data = {
    "arrival_params": {
        "calibration_period" : [calibration_start, calibration_end],

        "stationary_model": {
            "mu": arrival_mean,
            "sigma": arrival_std},

        "dynamic_model": {
            "reference_year": calibration_start,
            "slope": incoming_players_regression.slope,
            "intercept": incoming_players_regression.intercept,
            "sigma_residuals": residuals_std}

                    }
                }
            

config_path = "../config/simulation_params.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        params = json.load(f)
else:
    params = {}

params.update(arrival_data)

os.makedirs(os.path.dirname(config_path), exist_ok=True)

with open(config_path, "w") as f: # opening the file in writing mode
    json.dump(params, f, indent=3)

print("Both models' parameters have been saved in the .json file!")

## 2. Calibrating the Distribution of Intrinsic Potentials (Talent)

As seen in section 1.1, the distribution of players strengths maximum (intrinsic potential $P$) is stationary. We observed that the raw values are highly skewed, but that the transformation $Y=\ln(P)$ follows a "nice-shaped" distribution (distribution that could be recovered by a known distribution).

Let's look at what the distribution $Y$ looks like (using the data of the calibration period):

In [ ]:
log10_potential_mean = calibration_players["log10_strength"].mean()
log10_potential_std = calibration_players["log10_strength"].std()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plotting the logarithmic distribution
sns.histplot(calibration_players["log10_strength"], color="#bdc3c7", 
             alpha=0.5, edgecolor="white", stat="density", bins="auto", zorder=1,
             label="Log-observed Distribution")
sns.kdeplot(calibration_players["log10_strength"], color="black", alpha=0.4, linewidth=2.5, zorder=3, label="KDE", linestyle = ":")

# plot a vertical line for the mean value (logarithm!)
plt.axvline(log10_potential_mean, color="#e74c3c", linewidth=3, alpha=0.8, linestyle="--", zorder=2, 
            label=rf"Logarithmic Mean Value: $\mu_{{\log P}} = {log10_potential_mean:.2f}$")

plt.title(r"Distribution of the Logarithm of Player Potentials ($\log_{10} P$)", fontsize=25, weight="bold", pad=35)
plt.xlabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)
plt.ylabel("Density", fontsize=18)

sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

#### H1: Well-known distributions fitting

We will now calibrate this potential distribution by testing different probability density functions on the logarithmic data.

In [ ]:
# select possible distributions
# https://www.itl.nist.gov/div898/handbook/eda/section3/eda36.htm
from scipy import stats

distributions_dict = {"Normal": stats.norm,
                      "Skew Normal": stats.skewnorm,
                      "Laplace": stats.laplace,
                      "Gumbel (right)": stats.gumbel_r,
                      "Logistic": stats.logistic,
                      "Student's t": stats.t}

# calculate the parameters for the fit
#https://docs.scipy.org/doc/scipy-1.17.0/reference/generated/scipy.stats.fit.html
potential_params = {}
for name, distribution in distributions_dict.items():
    params = distribution.fit(calibration_players["log10_strength"]) # warning, returns: shapes, then loc and scale!
    potential_params[name] = {"loc": params[-2],
                              "scale": params[-1]}
    if len(params) > 2:
        potential_params[name].update({"shape": list(params[:-2])})

In [ ]:
# --- uncomment the code to save the potential parameters in the .json file ---

# potential_data = {"potential_params": potential_params}

# # saving the parameters in the .json file created earlier
# config_path = "../config/simulation_params.json"

# if os.path.exists(config_path):
#     with open(config_path, "r") as f: # opening the file in read mode
#         config_data = json.load(f)
# else:
#     config_data = {}

# if "potential_params" not in config_data:
#     config_data["potential_params"] = {}

# config_data["potential_params"].update(potential_data["potential_params"])

# os.makedirs(os.path.dirname(config_path), exist_ok=True)


# with open(config_path, "w") as f: # opening the file in writing mode
#     json.dump(config_data, f, indent=3)

In [ ]:
n_cols = 2
n_rows = (len(distributions_dict)+n_cols-1)//n_cols

plt.figure(figsize=(16, n_rows*5))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# defining the x range for the fit
x = np.linspace(calibration_players["log10_strength"].min(), 
                    calibration_players["log10_strength"].max(), 
                    1000)

for i, (name, distribution) in enumerate(distributions_dict.items()):
    plt.subplot(n_rows,n_cols,i+1)

    # get the parameters of the fit
    loc = potential_params[name]["loc"]
    scale = potential_params[name]["scale"]
    shape = potential_params[name].get("shape", [])

    # get the legend
    hist_legend = "Log-observed Distribution" if i==0 else None
    KDE_legend = "KDE" if i==0 else None
    fit_legend = "Theoretical Curve" if i==0 else None

    # get the probability density function
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html
    # * to unpack the elements in the list shape (if not empty)
    fit = distribution.pdf(x, *shape, loc=loc, scale=scale)

    # plotting the logarithmic distribution
    sns.histplot(calibration_players["log10_strength"], color="#bdc3c7", 
             alpha=0.5, stat="density", bins="auto",
             label=hist_legend)


    sns.kdeplot(calibration_players["log10_strength"], color="black", 
                alpha=0.4, linewidth=1.5, label=KDE_legend, linestyle=":")

    
    # plotting the theoritical curve (from the distributions chosen)
    plt.plot(x, fit, color="#2c3e50", linewidth=2.5, label=fit_legend)

    plt.title(f"{name} Distribution", fontsize=16, weight="bold")

    if i%n_cols == 0:
        plt.ylabel("Density")
    else:
        plt.ylabel("")
    
    if i+n_cols >= len(distributions_dict):
        plt.xlabel(r"$\log_{10} P$")
    else:
        plt.xlabel("")

    sns.despine()

plt.suptitle(r"Comparison of Theoretical Distributions for $\mathbf{log_{10} P}$", 
             fontsize=24, weight="bold", y=1.)
plt.figlegend(loc="lower center", ncols=3, fontsize=13, frameon=True, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# defining the x range for the fit
x = np.linspace(calibration_players["log10_strength"].min(), 
                    calibration_players["log10_strength"].max(), 
                    1000)

# plotting the logarithmic distribution
sns.histplot(calibration_players["log10_strength"], color="#bdc3c7", 
             alpha=0.5, edgecolor="white", stat="density", bins="auto", zorder=1,
             label="Log-observed Distribution")
sns.kdeplot(calibration_players["log10_strength"], color="black", alpha=0.4, linewidth=2.5, zorder=3, label="KDE", linestyle = ":")

skew_normal_params = potential_params["Skew Normal"]

skew_normal_fit = stats.skewnorm.pdf(x, skew_normal_params["shape"], loc=skew_normal_params["loc"], scale=skew_normal_params["scale"])

# plot of the skew normal fit
plt.plot(x, skew_normal_fit, color="#2c3e50", linewidth=4, label="Skew Normal Fit", zorder=2)


plt.suptitle(r"Comparison of Skewed Normal Distribution for $\mathbf{log_{10} P}$", 
             fontsize=24, weight="bold", y=1.)
plt.xlabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)
plt.ylabel("Density", fontsize=18)

sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()



We see that the fit are not really good, because some distributions are symmetric (like the normal distribution) while the data is skewed to the right. The best fit here is the skewed normal distribution. But the result is not perfect, so we will try to combine two Gaussian distributions (so 5 parameters: mean and std of each distribution, as well as weight of each distribution) to see if we can get a better fit.

#### H2: Mixture of 2 Distributions

To fit the parameters of a mixture of 2 distributions, we can use the method of maximum likelihood estimation (MLE). The idea is to find the parameters that maximize the likelihood of observing our data given the model. It will be done by minimizing the negative log-likelihood (equivalent to maximizing the likelihood). Let's try this approach for normal and skewed normal distributions.

Mathematicaly, let $y_i=\log_{10}(P_i)$ where $P_i$ is the potential of player $i$. The probability density function (pdf) of the mixture of 2 distributions for a given value $y_i$ is:

$$f(y_i|\theta) = w \cdot f_1(y_i | \theta_1) + (1-w) \cdot f_2(y_i | \theta_2)$$

where $w$ is the weight of the first distribution ($0<w<1$), $f_1$ and $f_2$ are the pdf of the first and second distribution respectively. $\theta_1$ and $\theta_2$ are the parameters of these distributions (mean and standard deviation for normal distributions).

The goal is now to minimize the negative log-likelihood of the observed data given this model. The likelihood function for a set of observed data points $Y = \{y_1, y_2, ..., y_n\}$ is:

$$L(\theta|Y) = \prod_{i=1}^n f(y_i|\theta)$$

Then, the negative log-likelihood is:

$$-\ln L(\theta|Y) = -\sum_{i=1}^n \ln f(y_i|\theta)$$

By minimizing this negative log-likelihood with respect to the parameters $\theta = \{w, \theta_1, \theta_2\}$, we can find the best-fitting parameters for our mixture model.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html
from scipy.optimize import minimize

In [ ]:
# function that returns for all x the value of the pdf of a mixture of 2 distributions with given parameters 
# calculates f(yi|parameters) for all yi in x
def normal_mixture_pdf(x, w, mean1, std1, mean2, std2):
    return w*stats.norm.pdf(x, loc=mean1, scale=std1) + (1-w)*stats.norm.pdf(x, loc=mean2, scale=std2)

def skewed_normal_mixture_pdf(x, w, mean1, std1, shape1, mean2, std2, shape2):
    return w*stats.skewnorm.pdf(x, a=shape1, loc=mean1, scale=std1) + (1-w)*stats.skewnorm.pdf(x, a=shape2, loc=mean2, scale=std2)

# function that calculates the  - log-likelihood of the data given the parameters of the mixture (to minimize it for the fit)
def neg_log_likelihood_normal(params):
    w, mean1, std1, mean2, std2 = params

    if w < 0 or w > 1 or std1 <= 0 or std2 <= 0: # to avoid invalid parameters
        return 1e9 # a very large number to be sure not finding the solution in this case
    
    pdf_values = normal_mixture_pdf(calibration_players["log10_strength"], w, mean1, std1, mean2, std2)
    log_likelihood_normal = np.sum(np.log(pdf_values))
    return -log_likelihood_normal

def neg_log_likelihood_skewed_normal(params):
    w, mean1, std1, shape1, mean2, std2, shape2 = params

    if w < 0 or w > 1 or std1 <= 0 or std2 <= 0: # to avoid invalid parameters
        return 1e9 # a very large number to be sure not finding the solution in this case
    

    pdf_values = skewed_normal_mixture_pdf(calibration_players["log10_strength"], w, mean1, std1, shape1, mean2, std2, shape2)
    neg_log_likelihood_skewed_normal = np.sum(np.log(pdf_values))
    return -neg_log_likelihood_skewed_normal

# minimization of the negative log-likelihood to find the best parameters for the normal mixture
# values for initial guess (mean initial value a bit left and right from the mean to help finding the 2 components)
mean_log_strength = np.mean(calibration_players["log10_strength"])
std_log_strength = np.std(calibration_players["log10_strength"])

minimization_normal = minimize(neg_log_likelihood_normal, 
                               [0.5, mean_log_strength-0.5, std_log_strength, mean_log_strength+0.5, std_log_strength],
                               method="Nelder-Mead")

minimization_skewed = minimize(neg_log_likelihood_skewed_normal, 
                               [0.5, mean_log_strength-0.2, std_log_strength, 1, mean_log_strength+0.2, std_log_strength, -1],
                               method="Nelder-Mead")

if (minimization_normal.success == True and minimization_skewed.success == True):
    print("Minimization of the negative log-likelihood for both mixtures was successful!")
else: 
    if minimization_normal.success == False:
        print("Minimization of the negative log-likelihood for the normal mixture failed:", minimization_normal.message)
    if minimization_skewed.success == False:
        print("Minimization of the negative log-likelihood for the skewed normal mixture failed:", minimization_skewed.message)

Let's have a look at a visualisation of the fit of the mixture of 2 Gaussian/skewed normal distributions.

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# defining the x range for the fit
x = np.linspace(calibration_players["log10_strength"].min(), 
                    calibration_players["log10_strength"].max(), 
                    1000)

# plotting the logarithmic distribution
sns.histplot(calibration_players["log10_strength"], color="#bdc3c7", 
             alpha=0.5, edgecolor="white", stat="density", bins="auto", zorder=1,
             label="Log-observed Distribution")
sns.kdeplot(calibration_players["log10_strength"], color="black", alpha=0.4, linewidth=2.5, zorder=3, label="KDE", linestyle = ":")

# plot the gaussian mix (individually + together)
w_normal, mean1_normal, std1_normal, mean2_normal, std2_normal = minimization_normal.x


Gaussian_1 = w_normal*stats.norm.pdf(x, loc=mean1_normal, scale=std1_normal)
Gaussian_2 = (1-w_normal)*stats.norm.pdf(x, loc=mean2_normal, scale=std2_normal)
Gaussian_mix = Gaussian_1 + Gaussian_2

plt.plot(x, Gaussian_1, label=f"Gaussian 1 ({w_normal*100:.1f}%)", linewidth=3, linestyle="--", color="#60a215e8", alpha=0.8)
plt.plot(x, Gaussian_2, label=f"Gaussian 2 ({(1-w_normal)*100:.1f}%)", linewidth=3, linestyle="--", color="#1484cfe8", alpha=0.8)
plt.plot(x, Gaussian_mix, label="Gaussian Combination Fit", linewidth=4, color="#2c3e50")


plt.suptitle(r"Comparison of Gaussian Mixture Model for $\mathbf{log_{10} P}$", 
             fontsize=24, weight="bold", y=1.)
plt.xlabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)
plt.ylabel("Density", fontsize=18)

sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# defining the x range for the fit
x = np.linspace(calibration_players["log10_strength"].min(), 
                    calibration_players["log10_strength"].max(), 
                    1000)

# plotting the logarithmic distribution
sns.histplot(calibration_players["log10_strength"], color="#bdc3c7", 
             alpha=0.5, edgecolor="white", stat="density", bins="auto", zorder=1,
             label="Log-observed Distribution")
sns.kdeplot(calibration_players["log10_strength"], color="black", alpha=0.4, linewidth=2.5, zorder=3, label="KDE", linestyle = ":")

# plot the gaussian mix (individually + together)
w_skewed, mean1_skewed, std1_skewed, shape1_skewed, mean2_skewed, std2_skewed, shape2_skewed = minimization_skewed.x


skewed_1 = w_skewed*stats.skewnorm.pdf(x, shape1_skewed, loc=mean1_skewed, scale=std1_skewed)
skewed_2 = (1-w_skewed)*stats.skewnorm.pdf(x, shape2_skewed, loc=mean2_skewed, scale=std2_skewed)
skewed_mix = skewed_1 + skewed_2

plt.plot(x, skewed_1, label=f"Skewed Normal 1 ({w_skewed*100:.1f}%)", linewidth=3, linestyle="--", color="#60a215e8", alpha=0.8)
plt.plot(x, skewed_2, label=f"Skewed Normal 2 ({(1-w_skewed)*100:.1f}%)", linewidth=3, linestyle="--", color="#1484cfe8", alpha=0.8)
plt.plot(x, skewed_mix, label="Skewed Normal Combination Fit", linewidth=4, color="#2c3e50")


plt.suptitle(r"Comparison of Skewed Normal Mixture Model for $\mathbf{log_{10} P}$", 
             fontsize=24, weight="bold", y=1.)
plt.xlabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)
plt.ylabel("Density", fontsize=18)

sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

The results are quite good, especially for the skewed normal distribution. It's much better than a single distribution, with one distribution fitting the main part in the middle, and the other distribution fitting the right tail.

#### H4: Non-parametric estimation (KDE)

The last possibility is to use directly the Kernel Density Estimation (KDE) to estimate the pdf of the data.

In [ ]:
from scipy.stats import gaussian_kde

kde = gaussian_kde(calibration_players["log10_strength"], bw_method="scott")

#### Saving the parameters 

In [ ]:
normal_mix_params = {
    "w": w_normal,
    "mean1": mean1_normal,
    "std1": std1_normal,
    "mean2": mean2_normal,
    "std2": std2_normal
}

skewed_mix_params = {
    "w": w_skewed,
    "mean1": mean1_skewed,
    "std1": std1_skewed,
    "shape1": shape1_skewed,
    "mean2": mean2_skewed,
    "std2": std2_skewed,
    "shape2": shape2_skewed
}

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        config_data = json.load(f)
else:
    config_data = {} 

if "potential_params" not in config_data:
    config_data["potential_params"] = {}

config_data["potential_params"]["normal_mixture"] = normal_mix_params
config_data["potential_params"]["skewed_normal_mixture"] = skewed_mix_params

with open(config_path, "w") as f:
    json.dump(config_data, f, indent=3)

The kde won't be saved directly in the .json file, but will be available by directly using the data in the .csv file and applying the KDE function (`gaussian_KDE`) on it.

## 3. Calibrating the Aging Curves (Evolution of Strength with Age)

## 4. Calibrating Career Duration and Retirement

## Sources

$\textbf{1.2. Stationarity of the Number of New Players (Quantity)}$

- Ghasemi A., & Zahediasl S. (2012). $\textit{Normality Tests for Statistical Analysis: A Guide for Non-Statisticians}$, International Journal of Endocrinology and Metabolism, 10(2), 486–489.
https://pmc.ncbi.nlm.nih.gov/articles/PMC3693611/ $\newline$
(for justifying the combination of graphic inspection + statistical tests for normality, because if statistical tests with a p-value only can lead to erroneous conclusions)
- Razali, N. M., & Wah, Y. B. (2011). $\textit{Power comparisons of Shapiro-Wilk, Kolmogorov-Smirnov, Lilliefors and Anderson-Darling tests.}$ Journal of Statistical Modeling and Analytics, 2(1), 21–33. https://www.nrc.gov/docs/ml1714/ml17143a100.pdf $\newline$ (for justifying the choice of Shapiro-Wilk test for normality, small samples)
- Shapiro, S. S., & Wilk, M. B. (1965). $\textit{An Analysis of Variance Test for Normality (Complete Samples).} Biometrika, 52(3/4), 591–611. https://doi.org/10.2307/2333709 $\\newline$
(for the original paper introducing the Shapiro-Wilk test)
- Wilk, M. B., & Gnanadesikan, R. (1968). Probability Plotting Methods for the Analysis of Data. Biometrika, 55(1), 1–17. https://doi.org/10.2307/2334448 $\\newline$
(for the original paper introducing the Q-Q plot)


